<a href="https://colab.research.google.com/github/alfiyatunnajah9/Tugas-Machine-Learning/blob/main/praktikum_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 13.11 Implementasi Python
## Praktikum 13.1 Persiapan Library

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import matplotlib.pyplot as plt

## Praktikum 13.2 Import Dataset

In [ ]:
import numpy as np

# Menggunakan dataset CIFAR-10 bawaan Keras
(x_train, y_train), (x_val, y_val) = keras.datasets.cifar10.load_data()

# Membatasi jumlah data agar proses training/resize tidak terlalu lama di Colab
x_train, y_train = x_train[:5000], y_train[:5000]
x_val, y_val = x_val[:1000], y_val[:1000]

# Parameter
IMG_SIZE = (160, 160)
BATCH_SIZE = 32
jumlah_kelas = 10

# Fungsi untuk resize citra karena MobileNetV2 membutuhkan input minimal 75x75 (disarankan 160x160)
def resize_and_preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    return image, label

# Membuat TensorFlow Dataset
train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.shuffle(buffer_size=1000).batch(BATCH_SIZE).map(resize_and_preprocess)

val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val))
val_ds = val_ds.batch(BATCH_SIZE).map(resize_and_preprocess)

  3235840/170498071 ━━━━━━━━━━━━━━━━━━━━ 1:02:47 23us/step

## Praktikum 13.3 Preprocessing

In [ ]:
train_ds = train_ds.map(lambda x, y: (preprocess_input(x),
    y))
val_ds = val_ds.map(lambda x, y: (preprocess_input(x), y))

## Praktikum 13.4 Training Model (Transfer Learning dengan MobileNetV2)

In [ ]:
base_model = MobileNetV2(input_shape=IMG_SIZE+(3,),
    include_top=False,weights="imagenet")
base_model.trainable = False  # Feature extraction: bekukan
  # bobot pre-trained

model = keras.Sequential([ base_model,
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dense(128,activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(jumlah_kelas,
    activation="softmax") ])

model.compile(optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])
history = model.fit(train_ds,validation_data=val_ds,
    epochs=15)

## Praktikum 13.5 Evaluasi

In [ ]:
val_loss, val_acc = model.evaluate(val_ds)
print(f"Akurasi Validasi: {val_acc:.3f}")

## Praktikum 13.6 Visualisasi

In [ ]:
plt.plot(history.history["accuracy"],
    label="Training Accuracy")
plt.plot(history.history["val_accuracy"],
    label="Validation Accuracy")
plt.xlabel("Epoch");
plt.ylabel("Akurasi")
plt.title("Kurva Akurasi - Transfer Learning MobileNetV2")
plt.legend()
plt.show()

## 13.12 Latihan Praktikum

Latih pada dataset CIFAR-10 (`keras.datasets.cifar10`) dan bandingkan dengan hasil Transfer Learning.

In [ ]:
model_cnn = keras.Sequential([ keras.layers.Conv2D(32,
    (3,3),activation="relu",input_shape=(32,32,3)),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Conv2D(64,(3,3),activation="relu"),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Flatten(),keras.layers.Dense(64,
    activation="relu"),keras.layers.Dense(10,
    activation="softmax") ])
model_cnn.compile(optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])
# Latih pada dataset CIFAR-10 (keras.datasets.cifar10) dan
# bandingkan dengan hasil Transfer Learning